In [ ]:
# Portfolio Mark-to-Market: values the swing deals in quotes_2.csv against one
# smoothed daily forward curve and reports per-deal MtM and monthly exposures.
# Reload the numba kernels FIRST - importlib.reload(storage_model) alone does not
# refresh storage_kernels (the @jit functions), which would mix old/new code.
import importlib
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

import storage_kernels
importlib.reload(storage_kernels)
import storage_model
importlib.reload(storage_model)

%matplotlib inline

In [ ]:
# Inputs - edit this cell

VAL_DATE       = pd.Timestamp("2010-08-19")   # valuation date (2010-08-19 = first quote with a full strip)
QUOTE_FILE     = "ttf q.xlsx"
PORTFOLIO_FILE = "quotes_2.csv"

# Model settings (per-deal vol and mean-reversion come from the portfolio file)
n_p_full      = 30    # price-tree half-width for the full stochastic run
clips_per_day = 1     # one clip per day -> clip size = daily volume
run_intrinsic = True  # also run the n_p=0 (intrinsic) pass per deal

# Quote rows before 2010-08-19 carry only 12 monthly contracts - too short for
# deals ending in 2013. Quote rows with fewer populated contracts than this are
# skipped when picking the valuation quote (the longest deal's backstop is
# Oct-2013; cell 4 re-validates curve coverage deal by deal).
MIN_STRIP_MONTHS = 43

In [ ]:
# Build ONE smoothed daily curve from the quote nearest <= VAL_DATE.
# The quote matrix is cached as parquet (read_excel ~1.7s, parquet ~30ms; the
# cache is rebuilt whenever the xlsx is newer). Monthly contracts are stepped
# to daily resolution (reindex + ffill), the DA quote anchors the curve at the
# quote date, and storage_model.smoothen_curve turns the steps into a smooth
# daily curve - the same construction as forward.ipynb's daily_curve_from_monthly.
quote_path = Path.cwd() / QUOTE_FILE
cache_path = quote_path.with_suffix(".parquet")
if cache_path.exists() and cache_path.stat().st_mtime >= quote_path.stat().st_mtime:
    quotes = pd.read_parquet(cache_path)
else:
    quotes = pd.read_excel(quote_path)
    quotes = quotes.rename(columns={quotes.columns[0]: "quote_date"})
    quotes = quotes.dropna(subset=["quote_date"]).copy()
    quotes["quote_date"] = pd.to_datetime(quotes["quote_date"], format="mixed")
    # Price columns can hold junk strings (e.g. "Retrieving...") - coerce to NaN.
    for c in quotes.columns[1:]:
        quotes[c] = pd.to_numeric(quotes[c], errors="coerce")
    quotes = quotes.sort_values("quote_date").reset_index(drop=True)
    quotes.to_parquet(cache_path)   # cache the CLEANED frame

contract_columns = sorted(
    [c for c in quotes.columns if re.fullmatch(r"TTFc\d+", str(c))],
    key=lambda c: int(re.search(r"\d+", str(c)).group()),
)

# Nearest quote <= VAL_DATE among rows whose strip is long enough to cover the
# portfolio horizon (cell 4 re-validates coverage deal by deal).
strip_len  = quotes[contract_columns].notna().sum(axis=1)
eligible   = quotes.loc[strip_len >= MIN_STRIP_MONTHS].reset_index(drop=True)
fd_quote   = storage_model.quote_row_for_fd_date(eligible, contract_columns, VAL_DATE)
quote_date = pd.Timestamp(fd_quote["quote_date"])

monthly, _ = storage_model.monthly_curve_from_quote(fd_quote, contract_columns)
day_index  = pd.date_range(monthly.index.min(), storage_model.month_end(monthly.index.max()), freq="D")
stepped    = monthly.reindex(day_index, method="ffill")

da_value = fd_quote.get("DA", np.nan)
if pd.notna(da_value):
    # Anchor the day-ahead quote at the quote date and ffill the gap to the front month.
    stepped = stepped.reindex(pd.date_range(quote_date, day_index.max(), freq="D"))
    stepped.loc[quote_date] = float(da_value)
    stepped = stepped.ffill()

daily_curve = storage_model.smoothen_curve(stepped).rename("value")

print(f"Quote date used: {quote_date:%Y-%m-%d} (VAL_DATE {VAL_DATE:%Y-%m-%d})")
print(f"Daily curve coverage: {daily_curve.index.min():%Y-%m-%d} .. {daily_curve.index.max():%Y-%m-%d} "
      f"({daily_curve.iloc[0]:.2f} .. {daily_curve.iloc[-1]:.2f} EUR/MWh)")

In [ ]:
# Load and clean the portfolio. quotes_2.csv is dirty: stray spaces in column
# names (" N_days "), numbers like " 2,400 " / "-1,200 ", and " -   " meaning 0.
# direction = sign of daily volume (-1 = sold/short deal): every deal is valued
# long with abs(daily volume); value/MtM/exposures are multiplied by direction.
def _num(x):
    s = str(x).strip().replace(",", "").replace(" ", "")
    if s in ("", "-", "nan"):
        return 0.0
    return float(s)

raw = pd.read_csv(PORTFOLIO_FILE)
raw.columns = [str(c).strip() for c in raw.columns]

port = pd.DataFrame({
    "product_type": raw["Product"].str.strip().str.lower().str.replace(r"\s+", "_", regex=True),
    "market":       raw["market"].str.strip(),
    "Start":        pd.to_datetime(raw["Start"].str.strip(), format="%d-%b-%y"),
    "End":          pd.to_datetime(raw["End"].str.strip(), format="%d-%b-%y"),
    "stock_days":   raw["current stock, days"].map(_num),
    "daily_vol":    raw["daily volume, Mwh"].map(_num),
    "n_days":       raw["N_days"].map(_num).astype(int),
    "executed":     raw["executed price, Eur/Mwh"].map(_num),
    "vol":          raw["vol"].map(_num),
    "sMR":          raw["MR"].map(_num),
    "strike":       raw["strike price, Eur/Mwh"].map(_num),
})
port.index = [f"deal_{i + 1}" for i in range(len(port))]

port["direction"]     = np.sign(port["daily_vol"]).astype(int)
port["abs_daily_vol"] = port["daily_vol"].abs()
port["capacity_mwh"]  = port["n_days"] * port["abs_daily_vol"]   # total obligation volume
port["stock_mwh"]     = port["stock_days"] * port["abs_daily_vol"]
port["premium_eur"]   = port["executed"] * port["capacity_mwh"]

# Per-deal sanity checks before valuing anything.
for name, d in port.iterrows():
    window = (d["End"] - d["Start"]).days + 1
    if window < d["n_days"]:
        raise ValueError(f"{name}: exercise window {window}d < N_days {d['n_days']}")
    backstop = (d["End"] + pd.DateOffset(months=1)).normalize() + pd.offsets.MonthEnd(0)
    if daily_curve.index.min() > VAL_DATE or daily_curve.index.max() < backstop:
        raise ValueError(f"{name}: daily curve does not cover {VAL_DATE:%Y-%m-%d}..{backstop:%Y-%m-%d}")
    if d["product_type"] == "put_swing" and d["stock_days"] < d["n_days"]:
        raise ValueError(f"{name}: put swing stock {d['stock_days']:.0f}d < N_days {d['n_days']}")
print("All deals pass window / curve-coverage / stock checks.")

display(port)

# Visual sanity check: the daily curve with each deal's exercise window shaded.
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(daily_curve.index, daily_curve.values, color="black", lw=1.2, label="daily forward curve")
for name, d in port.iterrows():
    ax.axvspan(d["Start"], d["End"], alpha=0.08,
               color="tab:blue" if d["product_type"] == "call_swing" else "tab:red")
ax.set_title(f"Daily curve ({quote_date:%Y-%m-%d}) with deal windows (blue = call swing, red = put swing)")
ax.set_ylabel("EUR/MWh")
ax.legend()
plt.show()

In [ ]:
# Value each deal: forced full exercise of N_days x |daily volume| MWh.
#  call swing - obligation to BUY gas at strike, choosing the highest-price
#               days; per-MWh payoff = price - strike. Modelled with the
#               withdraw machinery: starts full, terminal empty.
#  put swing  - obligation to SELL gas at strike, choosing the cheapest days;
#               per-MWh payoff = strike - price. Modelled with the inject
#               machinery: starts empty, terminal full.
# value_eur = s.v[0, s.n_p, init_inv] with init_inv = s.n_op_start (0 for put,
# n_states for call). Daily forward deltas (s.delta, MWh/day) and expected
# exercise (s.exp_ex) are direction-adjusted and kept for the exposure report.
# NB: the printed per-MWh intrinsic/extrinsic come from the result dict, which
# compares achieved price vs the flat window price WITHOUT netting the strike -
# with strike 20 vs a ~20 curve the intrinsic metric reads ~ -/+20 by design;
# value_eur is the strike-netted EUR value and is what feeds the MtM.
results = {}
deal_values = {}
delta_profiles = {}
exercise_profiles = {}

for name, d in port.iterrows():
    t0 = time.perf_counter()
    params = {
        "product_type":  d["product_type"],
        "valDate":       VAL_DATE,
        "storageStart":  d["Start"],
        "storageEnd":    d["End"],
        "vol":           d["vol"],
        "sMR":           d["sMR"],
        "n_p_full":      n_p_full,
        "run_intrinsic": run_intrinsic,
        "daily_max":     d["abs_daily_vol"],
        "clips_per_day": clips_per_day,
        "capacity_mwh":  d["capacity_mwh"],
        "strike":        d["strike"],
        "daily_curve":   daily_curve,
    }
    s, res = storage_model.run_valuation(None, params)

    init_inv  = s.n_op_start            # 0 for put swing, n_states for call swing
    value_eur = float(s.v[0, s.n_p, init_inv])

    results[name]     = res
    deal_values[name] = value_eur
    delta_profiles[name] = d["direction"] * pd.Series(
        np.asarray(s.delta[:s.n_t], dtype=float), index=s.date_span[:s.n_t])
    exercise_profiles[name] = d["direction"] * pd.Series(
        np.asarray(s.exp_ex[:s.n_t], dtype=float), index=s.date_span[:s.n_t])

    print(f"{name}: {d['product_type']:>10}  value_eur = {value_eur:>13,.0f}  "
          f"intrinsic = {res['intrinsic']:7.3f}  extrinsic = {res['extrinsic']:6.3f} EUR/MWh  "
          f"[{time.perf_counter() - t0:.2f}s]")

In [ ]:
# Mark-to-Market table. premium_eur = executed price x N_days x |daily volume|;
# value_eur (direction-adjusted) = direction x model value;
# mtm_eur = direction x (model value - premium). The sold deal's signs flip.
mtm = port[["product_type", "Start", "End", "daily_vol", "n_days", "strike", "executed"]].copy()
mtm["premium_eur"] = port["premium_eur"]
mtm["value_eur"]   = [port.loc[n, "direction"] * deal_values[n] for n in port.index]
mtm["mtm_eur"]     = [port.loc[n, "direction"] * (deal_values[n] - port.loc[n, "premium_eur"])
                      for n in port.index]

total = pd.Series({"product_type": "TOTAL",
                   "premium_eur": mtm["premium_eur"].sum(),
                   "value_eur":   mtm["value_eur"].sum(),
                   "mtm_eur":     mtm["mtm_eur"].sum()}, name="TOTAL")
mtm_table = pd.concat([mtm, total.to_frame().T])

fmt = mtm_table.copy()
for c in ["daily_vol", "n_days", "premium_eur", "value_eur", "mtm_eur"]:
    fmt[c] = fmt[c].map(lambda v: "" if pd.isna(v) else f"{v:,.0f}")
for c in ["Start", "End"]:
    fmt[c] = fmt[c].map(lambda v: "" if pd.isna(v) else f"{v:%Y-%m-%d}")
fmt = fmt.fillna("")
display(fmt)

In [ ]:
# Monthly forward exposure: each deal's direction-adjusted daily delta (MWh
# forward-equivalent per day) resampled to month sums; rows = months, one
# column per deal plus the PORTFOLIO total. The bar chart shows the portfolio
# exposure against the forward curve (twin axis).
monthly_exp = pd.DataFrame({n: p.resample("MS").sum() for n, p in delta_profiles.items()}).fillna(0.0)
monthly_exp = monthly_exp.loc[monthly_exp.abs().sum(axis=1) > 1e-9]   # drop all-zero months
monthly_exp["PORTFOLIO"] = monthly_exp.sum(axis=1)

fmt = monthly_exp.copy()
fmt.index = fmt.index.strftime("%Y-%m")
display(fmt.map(lambda v: f"{v:,.0f}"))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(monthly_exp.index, monthly_exp["PORTFOLIO"], width=20, color="tab:blue",
       alpha=0.7, label="portfolio exposure (MWh/month)")
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("MWh per month")
ax2 = ax.twinx()
ax2.plot(daily_curve.index, daily_curve.values, color="black", lw=1.0, label="forward curve")
ax2.set_ylabel("EUR/MWh")
ax.set_title("Portfolio monthly forward exposure vs forward curve")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="best")
plt.show()

# Consistency check: monthly sums per deal == total daily delta sum.
chk = pd.DataFrame({"daily_sum":   {n: p.sum() for n, p in delta_profiles.items()},
                    "monthly_sum": monthly_exp.drop(columns="PORTFOLIO").sum()})
assert np.allclose(chk["daily_sum"], chk["monthly_sum"], atol=1e-6), chk
print("Monthly exposure totals match daily delta sums.")